# Setting up encrypted inference

Encrypted models inference requires producing several artifacts to complete the porting to the encrypted world. Apart from exporting weights from the HuggingFace format to the CUDA one we need to:
- calibrate approximation
- create a plan for a fixed bts schedule.

| # | artifact | made by | ETA |
|---|---|---|---|
| 1 | `weights.bin.zip`, `client.npz` | `perseus.export` | seconds, CPU |
| 2 | calibration pool (tokens or images) | `perseus.calibrate.data` | needs network |
| 3 | `configs.json` — the fitted approximations | `perseus.calibrate` | ~5 min, GPU |
| 4 | the **plan** fix bootstraps schedule | `perseus.plan.plan_graph_dir` | ~2 min, CPU |

In [1]:
import os, json, time, subprocess, sys
from pathlib import Path

import numpy as np
import torch                                   # torch before _core (NCCL load order)
from omegaconf import OmegaConf

FAMILY  = os.environ.get("FAMILY", "bert")
REPO    = Path.cwd()
SCRATCH = os.environ["SCRATCH"]
PYTHON  = sys.executable
os.environ.setdefault("HF_HOME", f"{SCRATCH}/.cache")
os.environ.setdefault("HF_HUB_CACHE", f"{SCRATCH}/.cache")

FAMILIES = {
    "bert": dict(
        hf_name="textattack/bert-base-uncased-SST-2",
        hydra=dict(model="bert_base", dataset="sst2", approximation="bert_base"),
        calib_extra=["model.block_size=32"],   # == the deployed T (real arm caps at 32)
        configs_name="bert_base",
        graph_dir=".cache/graph_bert_base",
        plan_dir="bootstrap_placements/planned_bert_base",
        keep=("goldschmidt", "inv_sqrt_newton", ".var", "ln_affine", "remez"),  # BERT needs keep
        probe=["scripts/utils/bert_operating_point.py", "--block-size", "32"],
        notebook="bert_torch_forward.ipynb",
    ),
    "vit": dict(
        hf_name="google/vit-base-patch16-224",
        hydra=dict(model="vit_base_224", dataset="tiny_imagenet", approximation="vit_base"),
        calib_extra=["model.resolution=80"],
        configs_name="vit_base_80",
        graph_dir=".cache/graph_vit_base_80",
        plan_dir="bootstrap_placements/planned_vit_base_80",
        keep=(".var",),                        # relax — the shipped ViT recipe (2026-07-27)
        probe=None,
        notebook="vit_torch_forward.ipynb",
    ),
    "gpt2": dict(
        hf_name="openai-community/gpt2",
        hydra=dict(model="gpt2", dataset="openwebtext", approximation="gpt2_cutmax"),
        calib_extra=[],                        # block_size null = inferred from the config
        configs_name="gpt2_base",
        graph_dir=".cache/graph_gpt2_base",
        plan_dir="bootstrap_placements/planned_gpt2_base",
        keep=(),                               # decode = bare smart cut, no keeps
        probe=None,
        notebook="gpt2_torch_forward.ipynb",
    ),
}

F = FAMILIES[FAMILY]
MODEL_DIR = f"{SCRATCH}/.cache/perseus/models/{F['hf_name']}/classic"
CONFIGS   = f"{REPO}/configs/model/approximation/{F['configs_name']}/configs.json"
DATASET   = OmegaConf.load(REPO / f"perseus/configs/dataset/{F['hydra']['dataset']}.yaml")
run = lambda *a: subprocess.run([str(x) for x in a], check=True, cwd=REPO)

print(f"FAMILY = {FAMILY}   ({F['hf_name']})")
print(f"  weights -> {MODEL_DIR}")
print(f"  configs -> {CONFIGS}")
print(f"  dataset -> {DATASET.name} ({DATASET.kind})")

FAMILY = bert   (textattack/bert-base-uncased-SST-2)
  weights -> /leonardo_scratch/large/userexternal/azirilli/.cache/perseus/models/textattack/bert-base-uncased-SST-2/classic
  configs -> /leonardo_work/IscrC_eff-SAM2/azirilli/pycudafideslib/configs/model/approximation/bert_base/configs.json
  dataset -> sst2 (text)


## 1. Export the weights

Splits the checkpoint in two:
- `weights.bin.zip`, the tensors the server evaluates under encryption
- `client.npz`, what stays client-side in the clear (embeddings, and any head with no
  registered FHE approximation)

`export.adapter_for` dispatches on `config.model_type`, so this is the same call for
every architecture.

In [2]:
if not Path(f"{MODEL_DIR}/weights.bin.zip").exists():
    run(PYTHON, "-m", "perseus.export", "--model", F["hf_name"],
        "--out", str(Path(MODEL_DIR).parent))
else:
    print("already exported")

for f in ("weights.bin.zip", "client.npz"):
    p = Path(f"{MODEL_DIR}/{f}")
    print(f"  {f:<18} {p.stat().st_size/1e6:8.1f} MB")

already exported
  weights.bin.zip       315.9 MB
  client.npz             97.7 MB


## 2. Calibration pool

Samples of the same distribution the model will actually see, cached as an `.npy` next to the HF cache.

Building needs network.

In [3]:
from perseus.calibrate import data

if DATASET.kind == "image":
    pool = data.load_image_pool(F["hf_name"], DATASET)
    print(f"image pool: {pool.shape}")
else:
    pool = data.load_token_pool(F["hf_name"], DATASET)
    print(f"token pool: {len(pool):,} tokens")

token pool: 800,000 tokens


## 3. Calibrate

Fits every nonlinearity (LayerNorm inverse-sqrt, softmax, GELU) over the
model's ranges and writes `configs.json`.

`block_size` must an upper bound of the deployed sequence length: it sets the softmax denominator
range and the per-position LayerNorm rescale. The registry pins it per family.

In [4]:
if not Path(CONFIGS).exists():
    t0 = time.perf_counter()
    run(PYTHON, "-m", "perseus.calibrate",
        f"model={F['hydra']['model']}", f"dataset={F['hydra']['dataset']}",
        f"approximation={F['hydra']['approximation']}",
        *F["calib_extra"], f"calib_out_path={CONFIGS}")
    print(f"calibrated in {time.perf_counter()-t0:.0f} s")
else:
    print(f"already calibrated (delete {Path(CONFIGS).name} to refit)")

cfg = json.load(open(CONFIGS))
print(f"  {len(cfg['norm'])} norm, {len(cfg['softgelu'])} gelu, {len(cfg['softmax'])} softmax sites")

already calibrated (delete configs.json to refit)


  25 norm, 12 gelu, 12 softmax sites


### Check the operating point first

Runs the plaintext model on what the calibrator produced.

The first out-of-band site by depth is the block the encrypted run will die at. Cheaper
to find here than as a `Decode(): approximation error is too high` ten blocks in.

In [5]:
if F["probe"]:
    run(PYTHON, *F["probe"])
else:
    print(f"no probe wired for {FAMILY} — scripts/utils/bert_operating_point.py is the template")


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 37580.29it/s]
BertModel LOAD REPORT from: textattack/bert-base-uncased-SST-2
Key                      | Status     |  | 
-------------------------+------------+--+-
classifier.bias          | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 
classifier.weight        | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


/leonardo_work/IscrC_eff-SAM2/azirilli/he-aware-training/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


[probe] pool  /leonardo_scratch/large/userexternal/azirilli/.cache/perseus/pools/sst2_bert-base-uncased-SST-2_800000.npy  (800000 tokens)
[probe] SST-2 validation unavailable (Couldn't reach 'stanfordnlp/sst2' on the Hub (OfflineModeIsEnabled)); using --text only
[probe] gate text T=8 tokens=['[CLS]', 'a', 'masterpiece', 'of', 'modern', 'cinema', '.', '[SEP]']
[probe] framed regime: 1 sentences, lengths 8..8

NORM SITES — cx (max|x-mean|) and the inverse-sqrt input var_scaled = css[pos]*var
site                    c_scale  cx_pool   cx_run  cx_r/p   band_lo  band_hi    vs_pool     vs_run  vs_r/hi  verdict
transformer.h.0.ln_1     0.7769    14.49    12.52    0.86     0.005       10      1.672        1.3     0.13  
transformer.h.0.ln_2     0.1783    67.06    64.64    0.96     0.005       10      2.099      1.949     0.19  
transformer.h.1.ln_1      1.149    10.49    9.429    0.90     0.005       10       1.41      1.119     0.11  
transformer.h.1.ln_2     0.1638     72.9    68.13    0.93

## Run it

Artifacts done. The model can be run now, in eager mode. Running it belongs to the model
notebooks:

| FAMILY | notebook | |
|---|---|---|
| `bert` | `bert_torch_forward.ipynb` | sentence in, sentiment out |
| `vit` | `vit_torch_forward.ipynb` | image in, class out |
| `gpt2` | `gpt2_torch_forward.ipynb` | prefill, hand-off, generate |

## 4. Capture and plan, fix a bootstrap schedule (Optional, Slow)

Only buys speed: eager places bootstraps reactively, a plan places them deliberately.
BERT 96.5s -> 83.4s, ViT-80 118s -> 110s.

Needs a capture, that is one slow forward in `Sync` with full rotation keys, so it deserves its own job
(~35 min for BERT-base):

```bash
TASK=bert STAGE=capture GATE_BLOCKS=12 sbatch scripts/run_task.sh
```

Planning is then ~2 min on CPU.

In [6]:
from perseus.plan import PlanOptions, plan_graph_dir

graphs = sorted(Path(F["graph_dir"]).glob("block_*/graph.json"))
staged = f"{F['plan_dir']}_staged"

if not graphs:
    print(f"no capture at {F['graph_dir']}")
    print(f"  make one:  TASK=<task> STAGE=capture sbatch scripts/run_task.sh")
else:
    summaries = plan_graph_dir(F["graph_dir"], staged,
                               PlanOptions(erase_keep_steps=F["keep"]))

    def n_placements(d):
        return sum(len(json.load(open(p)).get("placements", []))
                   for p in sorted(Path(d).glob("block_*_placement.json")))

    eager = sum(sum(n["op_type"] == "auto_bootstrap" for n in json.load(open(g))["nodes"])
                for g in graphs)
    print(f"\n{len(summaries)} blocks -> {staged}")
    print(f"  placements {n_placements(staged)}   (capture fired {eager} eager auto-bootstraps)")

    live = Path(F["plan_dir"])
    if live.is_dir():
        print(f"  live plan {live.name}: {n_placements(live)} placements — compare before promoting")

out=bootstrap_placements/planned_bert_base_staged max_abs=50.0 max_level=24 cache_read_level=17
[block_0] -> bootstrap_placements/planned_bert_base_staged/block_0_placement.json (entry: capture-derived, bts_level=16)


[min_cut] erase-autobts: dropped 20 reactive auto_bootstrap (min-cut re-places, incl. out-of-band w/ in-band ancestor); kept 20 (unsafe |abs|>50.0 + precision-keep ['goldschmidt', 'inv_sqrt_newton', '.var', 'ln_affine', 'remez']) as deliberate
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6625 max_in=18 output_level=19
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6685 max_in=20 output_level=21
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_6690 max_in=21 output_level=22
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6733 max_in=23 output_level=24
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_9188 max_in=16 output_level=17
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_19964 max_in=19 output_level=20
[min_cut] per-op caps active on 5 nodes (level_hint)
[min_cut] Running layer-by-layer min-cut (L=8)...


[min_cut] weight levels: down=17, k=16, kv=16, ln_1.bias=18, ln_1.weight=17, ln_2.bias=22, ln_2.weight=21, out=20, q=16, up=18, v=16
[min_cut] mask levels: gelu.half=[23], ln.center.ln_1=[21], ln.center.ln_2=[18], ln.scalemask=[19, 22]
[min_cut] emit-deliberate: 1 forced bts on |abs|>50.0 (no range-safe alternative — lm_head/residual)
  bootstraps=33 placements=3
[block_1] -> bootstrap_placements/planned_bert_base_staged/block_1_placement.json (entry: --entry-level 22 --entry-deg 1, bts_level=16)


[min_cut] erase-autobts: dropped 20 reactive auto_bootstrap (min-cut re-places, incl. out-of-band w/ in-band ancestor); kept 20 (unsafe |abs|>50.0 + precision-keep ['goldschmidt', 'inv_sqrt_newton', '.var', 'ln_affine', 'remez']) as deliberate
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6625 max_in=18 output_level=19
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6685 max_in=20 output_level=21
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_6690 max_in=21 output_level=22
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6733 max_in=23 output_level=24
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_9188 max_in=16 output_level=17
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_19964 max_in=19 output_level=20
[min_cut] per-op caps active on 5 nodes (level_hint)
[min_cut] Running layer-by-layer min-cut (L=8)...


[min_cut] weight levels: down=17, k=17, kv=17, ln_1.bias=19, ln_1.weight=18, ln_2.bias=23, ln_2.weight=22, out=20, q=17, up=19, v=17
[min_cut] mask levels: gelu.half=[23], ln.center.ln_1=[21], ln.center.ln_2=[19], ln.scalemask=[20, 22]
[min_cut] emit-deliberate: 1 forced bts on |abs|>50.0 (no range-safe alternative — lm_head/residual)
  bootstraps=33 placements=3
[block_2] -> bootstrap_placements/planned_bert_base_staged/block_2_placement.json (entry: --entry-level 23 --entry-deg 1, bts_level=16)


...

[min_cut] erase-autobts: dropped 17 reactive auto_bootstrap (min-cut re-places, incl. out-of-band w/ in-band ancestor); kept 20 (unsafe |abs|>50.0 + precision-keep ['goldschmidt', 'inv_sqrt_newton', '.var', 'ln_affine', 'remez']) as deliberate


[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6625 max_in=18 output_level=19
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6685 max_in=20 output_level=21
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_6690 max_in=21 output_level=22
[min_cut] Zero-cost op shows level change; forcing cost=1. op=add output=v_6733 max_in=23 output_level=24
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_9188 max_in=19 output_level=20
[min_cut] Zero-cost op shows level change; forcing cost=1. op=negate output=v_19964 max_in=23 output_level=24
[min_cut] per-op caps active on 5 nodes (level_hint)
[min_cut] Running layer-by-layer min-cut (L=8)...


[min_cut] weight levels: down=17, k=17, kv=17, ln_1.bias=17, ln_1.weight=24, ln_2.bias=22, ln_2.weight=21, out=20, q=17, up=17, v=17
[min_cut] mask levels: gelu.half=[23], ln.center.ln_1=[21], ln.center.ln_2=[18], ln.scalemask=[19, 22]
  bootstraps=33 placements=3



12 blocks -> bootstrap_placements/planned_bert_base_staged
  placements 309   (capture fired 363 eager auto-bootstraps)


  live plan planned_bert_base: 309 placements — compare before promoting
